In [1]:
!pip install -q langchain langgraph langchain-groq pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.1 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

import pypdf

In [9]:
#creando variable de entorno
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [10]:
ruta_pdf = "/content/techshop-documentacion.pdf"

with open(ruta_pdf, "rb") as archivo:
    lector = pypdf.PdfReader(archivo)
    texto_documento = ""
    for pagina in lector.pages:
        texto_documento += pagina.extract_text() + "\n"



In [11]:
@tool
def buscar_en_documento(query: str) -> str:
    """
    Busca información relevante en el documento PDF de TechShop.
    Usa esta herramienta para responder cualquier pregunta sobre:
    - Políticas de reembolso y devoluciones
    - Preguntas frecuentes (FAQ)
    - Guía de envíos y entregas
    - Cualquier otra información contenida en el documento
    """
    # Dividir el texto en párrafos
    parrafos = [p for p in texto_documento.split("\n") if p.strip()]

    # Buscar párrafos que contengan palabras de la query
    palabras_query = query.lower().split()
    resultados = []

    for i, parrafo in enumerate(parrafos):
        parrafo_lower = parrafo.lower()
        coincidencias = sum(1 for palabra in palabras_query if palabra in parrafo_lower)
        if coincidencias > 0:
            # Incluir contexto (párrafo anterior y siguiente)
            inicio = max(0, i - 1)
            fin = min(len(parrafos), i + 2)
            contexto = "\n".join(parrafos[inicio:fin])
            resultados.append(contexto)

    if not resultados:
        return "No se encontró información relevante en el documento de TechShop."

    # Devolver los fragmentos más relevantes (máximo 3)
    return "\n\n---\n\n".join(resultados[:3])

In [12]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key=os.environ["GROQ_API_KEY"]
)

print("LLM de Groq inicializado correctamente")

LLM de Groq inicializado correctamente


In [13]:
system_prompt = """
Eres un asistente virtual de TechShop, una tienda online de productos electrónicos.
Tu única fuente de verdad es el resultado de la herramienta 'buscar_en_documento'.

Reglas:
1. SIEMPRE usa la herramienta 'buscar_en_documento' para responder.
2. No inventes información que no esté en el documento.
3. Si el documento no contiene la respuesta, dilo claramente.
4. Responde en español, de forma clara, amigable y concisa.
5. Cuando uses información del documento, menciona en qué sección se encuentra.
6. Si el usuario pregunta algo fuera del alcance del documento, indica amablemente que solo podés responder sobre temas de TechShop.
"""

agente = create_react_agent(
    model=llm,
    tools=[buscar_en_documento],
    prompt=system_prompt
)

print("Agente creado correctamente")

Agente creado correctamente


/tmp/ipykernel_573/2373640616.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(


In [15]:
pregunta = "¿Cuál es el plazo para devolver un producto?"

resultado = agente.invoke({
    "messages": [("user", pregunta)]
})

respuesta_final = resultado["messages"][-1].content

print("Usuario:", pregunta)
print("Agente:", respuesta_final)

Usuario: ¿Cuál es el plazo para devolver un producto?
Agente: El plazo para devolver un producto es de 30 días corridos desde la fecha de entrega, según se establece en la sección 1.1 de la política de reembolso y devoluciones.


In [16]:
pregunta1 = "¿Puedo devolver un producto si ya abrí el empaque?"

resultado1 = agente.invoke({
    "messages": [("user", pregunta1)]
})

print("Usuario:", pregunta1)
print("Agente:", resultado1["messages"][-1].content)

Usuario: ¿Puedo devolver un producto si ya abrí el empaque?
Agente: Lo siento, pero no puedo encontrar información en el documento sobre la devolución de productos ya abiertos. Es posible que la política de devolución de TechShop no permita la devolución de productos que ya han sido abiertos o utilizados. Te recomiendo verificar la sección de política de devolución en el documento para obtener más información.


In [17]:
pregunta2 = "¿Qué métodos de pago aceptan en TechShop?"

resultado2 = agente.invoke({
    "messages": [("user", pregunta2)]
})

print("Usuario:", pregunta2)
print("Agente:", resultado2["messages"][-1].content)

Usuario: ¿Qué métodos de pago aceptan en TechShop?
Agente: Lo siento, no he podido encontrar la información solicitada en el documento. La información sobre los métodos de pago aceptados en TechShop no se encuentra disponible en el documento proporcionado.


In [18]:
pregunta3 = "¿Cuánto cuesta el envío express y cuánto tarda?"

resultado3 = agente.invoke({
    "messages": [("user", pregunta3)]
})

print("Usuario:", pregunta3)
print("Agente:", resultado3["messages"][-1].content)

Usuario: ¿Cuánto cuesta el envío express y cuánto tarda?
Agente: Lo siento, pero no he podido encontrar la información solicitada en el documento. La información sobre el costo y el tiempo de entrega del envío express no se encuentra en el documento proporcionado. Si necesitas más ayuda o tienes alguna otra pregunta, no dudes en preguntar.


In [19]:
pregunta5 = "¿Cuál es la capital de Francia?"

resultado5 = agente.invoke({
    "messages": [("user", pregunta5)]
})

print("Usuario:", pregunta5)
print("Agente:", resultado5["messages"][-1].content)

Usuario: ¿Cuál es la capital de Francia?
Agente: Lo siento, pero como asistente virtual de TechShop, mi única fuente de información es el documento de TechShop, el cual no contiene información sobre geografía o capitales de países. Mi capacidad para responder se limita a temas relacionados con TechShop, como políticas de reembolso, preguntas frecuentes, guía de envíos y entregas, y cualquier otra información contenida en el documento de TechShop. Si tienes alguna pregunta sobre estos temas, estaré encantado de ayudarte.
